In [ ]:
#lOAD PACKAGES
import pandas as pd
import matplotlib.pyplot as plt
import os, sys
import plotly.express as px
# Import the processing module from the same folder
sys.path.append(os.path.join("..", "scripts", "analysis"))
from processing import load_solutions, add_kwargs_as_indices, combine_solutions, read_parquet_and_convert, add_fields
# import processing 
# from pivottablejs import pivot_ui
G_save = False

dim = (1000,500)
g_BLUE = "1616A7"
g_GREY = "#7F7F7F"

legend_attr = dict(
    x=0.5,
    y=-0.25,
    yanchor="bottom",
    xanchor="center",
    orientation="h"
)


In [ ]:
def create_envelope(s_ed, s_uc, group_by = ['configuration', 'µ', 'iteration', 'day', 'hour,', 'r_id']):
    # Copy the relevant columns from s_ed['storage']
    envelope = s_ed['storage'][group_by + ['SOE_MWh', 'envelope_up_MWh', 'envelope_down_MWh']].copy()

    # Perform the first left join
    envelope = envelope.merge(
        s_uc['storage'][[col for col in group_by if col != 'iteration'] + ['SOE_MWh', 'envelope_up_MWh', 'envelope_down_MWh']].rename(
            columns={'SOE_MWh': 'SOE_DA_MWh', 'envelope_up_MWh': 'envelope_up_DA_MWh', 'envelope_down_MWh': 'envelope_down_DA_MWh'}
        ),
        on=[col for col in group_by if col != 'iteration'],
        how='left'
    )
    
    # Perform the second left join
    envelope = envelope.merge(
        s_uc['storage_parameters'][['r_id', 'SOE_max_MWh', 'initial_energy_proportion']].drop_duplicates(),
        on='r_id',
        how='left'
    )
    
    # Calculate initial state of energy (SOE) based on maximum SOE and initial energy proportion
    envelope['SOE_0_MWh'] = envelope['SOE_max_MWh'] * envelope['initial_energy_proportion']
    envelope['SOC'] = envelope['SOE_MWh'] / envelope['SOE_max_MWh'] 
    # Group by day, configuration, and resource ID, and get the last entry for each group

    return envelope

In [ ]:
ss = [
    # {'solution_folder': f"RTS-GMLC_v3.1.1s", 'VLGEN': 30, 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v4.1.1s", 'VLGEN': 30, 'model_type' : 'e-reserve'},
    {'solution_folder': f"RTS-GMLC_v32.3s", 'model_type' : 'envelope'},
    {'solution_folder': f"RTS-GMLC_v32.1s", 'model_type' : 'e-reserve'},
    # {'solution_folder': f"RTS-GMLC_v_s1.2s", 'VLGEN': 1e3, 'model_type' : 'stochastic'}
]
days = [2]
s_uc = []
s_ed = []
gcd_KPI_adequacy = []
gcdi_KPI_adequacy = []
solution_keys = ['storage_parameters','storage', 'dual_variables']
for sol in ss:
    # ρ = sol['ρ']
    s = sol['solution_folder']
    # s_uc_name = 's_uc' if sol['model_type'] == 'stochastic' else 's_uc'
    # s_ed_name = 's_sed'
    s_uc_ = load_solutions("s_uc", os.path.join("..", "output", s), days, solution_keys = solution_keys,  model_type = sol['model_type'], solution_id = s)
    if sol['model_type'] != 'stochastic':
        s_ed_ = load_solutions("s_ed", os.path.join("..", "output", s), days, solution_keys = solution_keys, model_type = sol['model_type'], solution_id = s)
    else:
        s_ed_ = load_solutions("s_suc", os.path.join("..", "output", s), days, solution_keys = solution_keys, model_type = sol['model_type'], solution_id = s)
    s_uc.append(s_uc_)
    s_ed.append(s_ed_)

    # gcd_KPI_adequacy_ = read_parquet_and_convert( os.path.join("..", "output", s, "all_gcd_KPI_adequacy.parquet"))
    # gcd_KPI_adequacy_ = add_fields(gcd_KPI_adequacy_, model_type = sol['model_type'], ρ=ρ, solution_id = s) 

    # gcdi_KPI_adequacy_ = read_parquet_and_convert( os.path.join("..", "output", s, "all_gcdi_KPI_adequacy.parquet"))
    # gcdi_KPI_adequacy_ = add_fields(gcdi_KPI_adequacy_, model_type = sol['model_type'], ρ=ρ, solution_id = s)

    # gcd_KPI_adequacy.append(gcd_KPI_adequacy_)
    # gcdi_KPI_adequacy.append(gcdi_KPI_adequacy_)

s_uc = combine_solutions(s_uc)
s_ed = combine_solutions(s_ed)
# gcd_KPI_adequacy = pd.concat(gcd_KPI_adequacy)
# gcdi_KPI_adequacy = pd.concat(gcdi_KPI_adequacy)

for k,v in s_uc.items():
    if 'µ' in v.columns:
        s_uc[k]['model_type'] =  v.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1) else x['model_type'], axis=1)
for k,v in s_ed.items():
    if 'µ' in v.columns:
        s_ed[k]['model_type'] = v.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1) else x['model_type'], axis=1)

# if 'µ' in gcdi_KPI_adequacy.columns: 
# #         # out['non-conservative'] = out[['model_type', 'µ']].apply(lambda x: (x[0] !='envelope') + (x[0] =='envelope')*(x[1]<1), axis = 1)
#     gcdi_KPI_adequacy['model_type'] = gcdi_KPI_adequacy.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1) else x['model_type'], axis=1)
#     gcd_KPI_adequacy['model_type'] = gcd_KPI_adequacy.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1) else x['model_type'], axis=1)




In [ ]:
# s_ed['dual_variables'].to_csv("s_ed_dual_variables.csv")

In [ ]:
# aux = s_uc['objective_function']
# aux.loc[aux.day == 1]
# (aux.loc[aux.day == 1].reserve_cost).sum()

In [ ]:
# group_by = ['hour','ρ','model_type']
# envelope = create_envelope(s_ed, s_uc, group_by = group_by)

da_SOE =  s_uc['storage'][['hour','day','model_type', 'hour_i','r_id','SOE_MWh', 'envelope_up_MWh', 'envelope_down_MWh']].copy()
da_SOE = da_SOE.merge(
        s_uc['storage_parameters'][['r_id', 'SOE_max_MWh']].drop_duplicates(),
        on=['r_id'],
        how='left',
    )

# da_SOE = da_SOE.loc[da_SOE.r_id.isin([102])]
# da_SOE = da_SOE.groupby(['hour', 'hour_i', 'ρ', 'model_type']).sum(min_count=1).reset_index()
# da_SOE = da_SOE.loc[da_SOE.r_id == 101]
da_SOE = da_SOE.groupby(['hour','day','r_id','model_type']).agg({'SOE_max_MWh':'max','SOE_MWh':'max','envelope_up_MWh': 'max', 'envelope_down_MWh': 'min'})
da_SOE = da_SOE.groupby(['hour','day', 'model_type']).sum().reset_index()
# da_SOE = da_SOE.fillna(0).groupby(['hour', 'hour_i', 'day', 'model_type'], as_index=False).sum()



da_SOE['envelope_down'] = da_SOE['envelope_down_MWh'] / da_SOE['SOE_max_MWh']
da_SOE['envelope_up'] = da_SOE['envelope_up_MWh'] / da_SOE['SOE_max_MWh']
da_SOE['SOC'] = da_SOE['SOE_MWh'] / da_SOE['SOE_max_MWh']

# da_SOE['hour'] = da_SOE['hour'] - (da_SOE['day']-1)*24

In [ ]:
fig = px.line(
    da_SOE,
    x='hour',
    y=['envelope_down_MWh', 'envelope_up_MWh'],
    # color_discrete_map={
    #     'envelope_down_MWh': 'blue',
    #     'envelope_up_MWh': 'purple'
    # },
    # line_dash='variable',
    color='model_type',
    facet_col = 'day',
    # facet_row = 'r_id',
    # line_group = 'ρ',
    # line_dash ='ρ',
    category_orders={"model_type": ["conservative", "envelope", "e-reserve"]},  # Explicitly define the order
    markers=True  # Adds markers to the lines
)
legend_attr = dict(
    x=0.5,
    y=-0.25,
    yanchor="bottom",
    xanchor="center",
    orientation="h"
)


fig.update_layout(
    plot_bgcolor="rgba(0,0,0,0)",
    yaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title='SOE [MWh]'),
    xaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title="hour"),
    width=dim[0],
    height=dim[1],
    showlegend=True,
    legend=legend_attr  # Include legend attributes
)

# Add outline boxes to each subplot
for axis in fig.layout:
    if axis.startswith('xaxis') or axis.startswith('yaxis'):
        fig.layout[axis].update(showline=True, linecolor="grey", mirror=True)

# Show the plot

# Increase figure size
fig.show()

In [ ]:
envelope_dual = s_uc['dual_variables'][['hour','day','r_id','model_type','dual_SOE_up_max_MU_MW', 'dual_SOE_down_max_MU_MW', 'dual_SOE_up_min_MU_MW', 'dual_SOE_down_min_MU_MW']].copy()
# envelope_dual = envelope_dual.fillna(0)
envelope_dual['r_id'] = envelope_dual.r_id.fillna(0) 

# envelope_dual['hour'] = envelope_dual['hour'] - (envelope_dual['day']-1)*24

In [ ]:
ramp_dual = s_uc['dual_variables'][['hour','day','r_id','model_type','dual_ramp_up_thermal_MU_MW', 'dual_ramp_down_thermal_MU_MW', 'dual_ramp_up_nonthermal_MU_MW', 'dual_ramp_down_nonthermal_MU_MW']].copy()
ramp_dual['r_id'] = ramp_dual.r_id.fillna(0) 
ramp_dual_ed = s_ed['dual_variables'][['hour','day','r_id','model_type','dual_ramp_up_thermal_MU_MW', 'dual_ramp_down_thermal_MU_MW', 'dual_ramp_up_nonthermal_MU_MW', 'dual_ramp_down_nonthermal_MU_MW']].copy()
ramp_dual_ed['r_id'] = ramp_dual_ed.r_id.fillna(0) 



In [ ]:
# Sort envelope_dual and add formatting for the legend
# envelope_dual['r_id'] = envelope_dual['r_id'].astype(int)
# envelope_dual.sort_values(['hour', 'r_id', 'model_type'], inplace=True)
fig = px.line(
    envelope_dual.melt(id_vars = ['hour', 'day', 'r_id', 'model_type'], value_vars   = ['dual_SOE_up_max_MU_MW', 'dual_SOE_down_max_MU_MW', 'dual_SOE_up_min_MU_MW', 'dual_SOE_down_min_MU_MW']),
    x='hour',
    y='value',
    line_group='variable',
    color  = 'r_id',
    facet_col='model_type',
    # color = 'r_id',
    category_orders={"model_type": ["conservative", "envelope", "e-reserve"]},  # Explicitly define the order
    markers=True  # Adds markers to the lines
)

# fig.update_layout(
#     plot_bgcolor="rgba(0,0,0,0)",
#     yaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title='dual envelope'),
#     xaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title="hour"),
#     width=dim[0],
#     height=dim[1],
#     showlegend=True,
#     # legend=legend_attr  # Include legend attributes
# )

# # Add outline boxes to each subplot
# for axis in fig.layout:
#     if axis.startswith('xaxis') or axis.startswith('yaxis'):
#         fig.layout[axis].update(showline=True, linecolor="grey", mirror=True)

# Show the plot
fig.show()

In [ ]:
fig = px.line(
    ramp_dual.melt(id_vars = ['hour', 'day', 'r_id', 'model_type'], value_vars = ['dual_ramp_up_thermal_MU_MW', 'dual_ramp_down_thermal_MU_MW', 'dual_ramp_up_nonthermal_MU_MW', 'dual_ramp_down_nonthermal_MU_MW']),
    x='hour',
    y='value',
    line_group='variable',
    color  = 'r_id',
    facet_col='model_type',
    # color = 'r_id',
    category_orders={"model_type": ["conservative", "envelope", "e-reserve"]},  # Explicitly define the order
    markers=True  # Adds markers to the lines
)
fig.show()

In [ ]:
fig = px.line(
    ramp_dual_ed.melt(id_vars = ['hour', 'day', 'r_id', 'model_type'], value_vars = ['dual_ramp_up_thermal_MU_MW', 'dual_ramp_down_thermal_MU_MW', 'dual_ramp_up_nonthermal_MU_MW', 'dual_ramp_down_nonthermal_MU_MW']),
    x='hour',
    y='value',
    line_group='variable',
    color  = 'r_id',
    facet_col='model_type',
    # color = 'r_id',
    category_orders={"model_type": ["conservative", "envelope", "e-reserve"]},  # Explicitly define the order
    markers=True  # Adds markers to the lines
)
fig.show()

In [ ]:
# Create the plot using Plotly Express
da_mp = s_uc['dual_variables'][['hour','day','dual_supply_demand_balance_MU_MW','model_type',]].copy()
da_mp = da_mp.dropna()
da_mp['hour'] = da_mp['hour'] - (da_mp['day']-1)*24
da_mp.sort_values(by=['model_type', 'day', 'hour'], inplace=True)
# da_mp['hour'] = da_mp['hour'].astype(str).str.zfill(2)  # Ensure hour is a string with leading zeros
da_mp['hour'] = da_mp['hour'].astype(int) - da_mp['hour'].astype(int).min()
# da_mp_st = da_mp[da_mp['model_type'] == 'stochastic']
# da_mp = da_mp[da_mp['model_type'] != 'stochastic']

fig = px.line(
    da_mp,
    x='hour',
    y='dual_supply_demand_balance_MU_MW',
    facet_col='day',
    color='model_type',
    line_dash ='model_type',
    # symbol ='ρ',
    # color ='model_type',
    # line_group='model_type',
    category_orders={"model_type": ["conservative", "envelope", "e-reserve"]},  # Explicitly define the order
    # labels={'dual_supply_demand_balance_MU_MW': 'Dual Variable', 'hour': 'Hour'},
    markers=True  # Adds markers to the lines
)

fig.update_layout(
    plot_bgcolor="rgba(0,0,0,0)",
    yaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title='DA energy marginal price [$/MWh]'),
    xaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title="hour"),
    width=dim[0],
    height=dim[1],
    showlegend=True,
    legend=legend_attr  # Include legend attributes
)

# Add outline boxes to each subplot
for axis in fig.layout:
    if axis.startswith('xaxis') or axis.startswith('yaxis'):
        fig.layout[axis].update(showline=True, linecolor="grey", mirror=True)

# Show the plot
fig.show()


In [ ]:
if G_save:
    fig.write_image("da_e_mp_hour.pdf", width=dim[0], height=dim[1])

In [ ]:
# Create the plot using Plotly Express
rt_mp = s_ed['dual_variables'][['hour','day','dual_supply_demand_balance_MU_MW','model_type','iteration']].copy()
rt_mp.sort_values(by=['model_type', 'day', 'hour'], inplace=True)
rt_mp = rt_mp.dropna()
# rt_mp['hour'] = rt_mp['hour'].astype(str).str.zfill(2)  # Ensure hour is a string with leading zeros
rt_mp['hour'] = rt_mp['hour'] - (rt_mp['day']-1)*24


In [ ]:

fig = px.line(
    rt_mp,
    x='hour',
    y='dual_supply_demand_balance_MU_MW',
    facet_col='day',
    color='model_type',
    line_dash = 'model_type',
    # facet_row='iteration',  # Add grouping based on 'iteration'
    category_orders={"model_type": ["conservative", "envelope", "e-reserve", 'stochastic']},  # Explicitly define the order
    # labels={'dual_supply_demand_balance_MU_MW': 'Dual Variable', 'hour': 'Hour'},
    markers=True,  # Adds markers to the lines
    line_group='iteration',  # Ensures lines are plotted independently for each 'iteration'
    # color = 'iteration'
)


fig.update_layout(
    plot_bgcolor="rgba(0,0,0,0)",
    yaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title='RT energy marginal price [$/MWh]'),
    xaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title="hour"),
    width=dim[0],
    height=dim[1],
    showlegend=True,
    legend=legend_attr  # Include legend attributes
)

# Add outline boxes to each subplot
for axis in fig.layout:
    if axis.startswith('xaxis') or axis.startswith('yaxis'):
        fig.layout[axis].update(showline=True, linecolor="grey", mirror=True)

# Show the plot
fig.show()


In [ ]:
if G_save:
    fig.write_image("rt_e_mp_hour.pdf", width=dim[0], height=dim[1])

In [ ]:
da_rmp = s_uc['dual_variables'][['hour','day','dual_reserve_up_requirement_MU_MW','dual_reserve_down_requirement_MU_MW','model_type']].dropna().copy()
da_ermp = s_uc['dual_variables'][['hour','hour_i','day','dual_energy_reserve_up_requirement_MU_MW','dual_energy_reserve_down_requirement_MU_MW','model_type']].dropna().copy()

da_ermp = da_ermp.groupby(['hour','day','model_type']).agg({'dual_energy_reserve_up_requirement_MU_MW': 'max', 'dual_energy_reserve_down_requirement_MU_MW': 'max'}).reset_index()

da_ermp.rename(columns={'dual_energy_reserve_up_requirement_MU_MW': 'dual_reserve_up_requirement_MU_MW', 'dual_energy_reserve_down_requirement_MU_MW': 'dual_reserve_down_requirement_MU_MW'}, inplace=True)
da_rmp = pd.concat([da_rmp, da_ermp], ignore_index=True)
da_rmp.sort_values(by=['hour','model_type', 'day', ], inplace=True)
da_rmp['hour'] = da_rmp['hour'] - (da_rmp['day']-1)*24
da_rmp = da_rmp[da_rmp['model_type'] != 'stochastic']

In [ ]:




fig = fig = px.line(
    da_rmp,
    x='hour',
    y='dual_reserve_up_requirement_MU_MW',
    facet_col='day',
    color='model_type',
    line_dash ='model_type',
    category_orders={"model_type": ["conservative","envelope", "e-reserve"]},  # Explicitly define the order
    # labels={'dual_supply_demand_balance_MU_MW': 'Dual Variable', 'hour': 'Hour'},
    markers=True  # Adds markers to the lines
)

legend_attr = dict(
    x=0.5,
    y=-0.25,
    yanchor="bottom",
    xanchor="center",
    orientation="h"
)

fig.update_layout(
    plot_bgcolor="rgba(0,0,0,0)",
    yaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title='DA reserve up price [$/MWh]'),
    xaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title="hour"),
    width=dim[0],
    height=dim[1],
    showlegend=True,
    legend=legend_attr  # Include legend attributes
)

# Add outline boxes to each subplot
for axis in fig.layout:
    if axis.startswith('xaxis') or axis.startswith('yaxis'):
        fig.layout[axis].update(showline=True, linecolor="grey", mirror=True)

# Show the plot
fig.show()


In [ ]:
if G_save:
    fig.write_image("da_rup_mp_hour.pdf", width=dim[0], height=dim[1])

In [ ]:

fig = fig = px.line(
    da_rmp,
    x='hour',
    y='dual_reserve_down_requirement_MU_MW',
    facet_col='day',
    color='model_type',
    line_dash ='model_type',
    category_orders={"model_type": ["conservative","envelope", "e-reserve"]},  # Explicitly define the order
    # labels={'dual_supply_demand_balance_MU_MW': 'Dual Variable', 'hour': 'Hour'},
    markers=True  # Adds markers to the lines
)

legend_attr = dict(
    x=0.5,
    y=-0.25,
    yanchor="bottom",
    xanchor="center",
    orientation="h"
)

fig.update_layout(
    plot_bgcolor="rgba(0,0,0,0)",
    yaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title='DA reserve down price [$/MWh]'),
    xaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title="hour"),
    width=dim[0],
    height=dim[1],
    showlegend=True,
    legend=legend_attr  # Include legend attributes
)

# Add outline boxes to each subplot
for axis in fig.layout:
    if axis.startswith('xaxis') or axis.startswith('yaxis'):
        fig.layout[axis].update(showline=True, linecolor="grey", mirror=True)

# Show the plot
fig.show()


In [ ]:
if G_save:
    fig.write_image("da_rdown_mp_hour.pdf", width=dim[0], height=dim[1])

In [ ]:
s_uc['dual_variables']